<a href="https://colab.research.google.com/github/Anurag-Anand0308/car-prize-detection-using-linear-regression/blob/main/car_prize_detection_using_linear_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import seaborn as sb
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor

In [2]:
#importing database and checking all columns
data = pd.read_csv('data.csv')

In [3]:
data[data['Engine Cylinders']>8]

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
460,Ferrari,456M,2001,premium unleaded (required),442.0,12.0,AUTOMATIC,rear wheel drive,2.0,"Exotic,High-Performance",Compact,Coupe,14,9,2774,223970
461,Ferrari,456M,2001,premium unleaded (required),442.0,12.0,MANUAL,rear wheel drive,2.0,"Exotic,High-Performance",Compact,Coupe,15,9,2774,219775
462,Ferrari,456M,2002,premium unleaded (required),442.0,12.0,AUTOMATIC,rear wheel drive,2.0,"Exotic,High-Performance",Compact,Coupe,14,9,2774,228625
463,Ferrari,456M,2002,premium unleaded (required),442.0,12.0,MANUAL,rear wheel drive,2.0,"Exotic,High-Performance",Compact,Coupe,15,9,2774,224585
464,Ferrari,456M,2003,premium unleaded (required),442.0,12.0,AUTOMATIC,rear wheel drive,2.0,"Exotic,High-Performance",Compact,Coupe,14,9,2774,228625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11394,Aston Martin,Virage,2012,premium unleaded (required),490.0,12.0,AUTOMATIC,rear wheel drive,2.0,"Exotic,High-Performance",Midsize,Coupe,18,13,259,208295
11395,Aston Martin,Virage,2012,premium unleaded (required),490.0,12.0,AUTOMATIC,rear wheel drive,2.0,"Exotic,High-Performance",Midsize,Convertible,18,13,259,223295
11448,Rolls-Royce,Wraith,2014,premium unleaded (required),624.0,12.0,AUTOMATIC,rear wheel drive,2.0,"Exotic,Luxury,High-Performance",Large,Coupe,21,13,86,284900
11449,Rolls-Royce,Wraith,2015,premium unleaded (required),624.0,12.0,AUTOMATIC,rear wheel drive,2.0,"Exotic,Luxury,High-Performance",Large,Coupe,21,13,86,294025


In [4]:
# Get unique values in 'Engine Cylinders' column
data['Engine Cylinders'].value_counts()

,count
Engine Cylinders,
4.0,4752
6.0,4489
8.0,2031
12.0,230
5.0,225
10.0,68
0.0,56
3.0,30
16.0,3


In [5]:
data = data[data['Engine Cylinders']!=3]
data.shape

(11884, 16)

In [6]:
data = data[data['Transmission Type'] != 'UNKNOWN']
data.shape

(11865, 16)

In [7]:
#droping all column and seeing the result
marketing = data['Market Category'].str.get_dummies(sep=',')
data['Age'] = 2025 - data['Year']

In [8]:
data['Average Mgp'] = data['highway MPG']+data['city mpg']
data['Average Mgp'] = data['Average Mgp']/2
data.head()

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP,Age,Average Mgp
0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135,14,22.5
1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650,14,23.5
2,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350,14,24.0
3,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450,14,23.0
4,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500,14,23.0


In [9]:
#features addition
features = ['Year', 'Engine HP', 'Engine Cylinders', 'Transmission Type',
            'Driven_Wheels', 'Age', 'Average Mgp', 'Popularity']
target = 'MSRP'

In [10]:
X = pd.concat([data[features], marketing], axis=1)
y = np.log1p(data[target])  # Log transformation

In [11]:
numerical_features = ['Year', 'Engine HP', 'Engine Cylinders', 'Age',
                      'Average Mgp', 'Popularity']
categorical_features = ['Transmission Type', 'Driven_Wheels']

In [12]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [13]:
X.shape

(11865, 18)

In [14]:
y.shape

(11865,)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
X_train.shape

(9492, 18)

In [17]:
X_test.shape

(2373, 18)

In [18]:
preprocessor.fit_transform(X_test).shape

(2373, 14)

In [19]:
preprocessor.fit_transform(X_train).shape

(9492, 14)

In [20]:
model = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=5)

In [21]:
model.fit(preprocessor.fit_transform(X_train),y_train)

GradientBoostingRegressor(learning_rate=0.05, max_depth=5, n_estimators=500)

In [22]:
y_pred = model.predict(preprocessor.fit_transform(X_test))

In [23]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R² Score: {r2:.2f}")


Mean Squared Error: 0.03
R² Score: 0.98


In [24]:
model.score(preprocessor.fit_transform(X_test),y_test)

0.9783142448452877